# Composite MNIST multi-label classifier

This notebook clones the repository, reads both datasets from Google Drive, concatenates only their training splits, and evaluates validation/test splits separately.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Configuration

Change `REPO_URL` and the three Google Drive paths before running the remaining cells. Each data directory must contain `train.pt`, `val.pt`, and `test.pt`.

In [ ]:
REPO_URL = 'https://github.com/your-username/your-repository.git'
BRANCH = 'main'
REPO_DIR = '/content/cnn'

ORIGINAL_DATA_DIR = '/content/drive/MyDrive/mnist_classifier_data/original'
BBOX_DATA_DIR = '/content/drive/MyDrive/mnist_classifier_data/uni_with_bboxes'
OUTPUT_DIR = '/content/drive/MyDrive/mnist_classifier_outputs/small'

MODEL_SIZE = 'small'  # 'small', 'large', or 'dense_net'
EPOCHS = 50
BATCH_SIZE = 256  # Use 32 for dense_net if accelerator memory is limited
NUM_WORKERS = 2
RESUME_CHECKPOINT = ''  # Optional path to last.pt on Drive

assert MODEL_SIZE in {'small', 'large', 'dense_net'}

In [ ]:
from pathlib import Path
import os
import subprocess

if 'your-username/your-repository' in REPO_URL:
    raise ValueError('Replace REPO_URL with your GitHub repository URL first.')

repo_path = Path(REPO_DIR)
if (repo_path / '.git').is_dir():
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
subprocess.run(['python', '-m', 'pip', 'install', '-q', 'matplotlib', 'tqdm'], check=True)
print(f'Repository ready at {Path.cwd()}')

In [ ]:
required = []
for directory in (Path(ORIGINAL_DATA_DIR), Path(BBOX_DATA_DIR)):
    for split in ('train', 'val', 'test'):
        path = directory / f'{split}.pt'
        if not path.is_file():
            required.append(str(path))

if required:
    raise FileNotFoundError('Missing dataset files:\n' + '\n'.join(required))
print('All six dataset files are available.')

In [ ]:
import sys

command = [
    sys.executable, '-m', 'src_model_cls.train',
    '--original-data-dir', ORIGINAL_DATA_DIR,
    '--bbox-data-dir', BBOX_DATA_DIR,
    '--output-dir', OUTPUT_DIR,
    '--model-size', MODEL_SIZE,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--num-workers', str(NUM_WORKERS),
]
if RESUME_CHECKPOINT:
    command.extend(['--resume', RESUME_CHECKPOINT])

print(' '.join(command))
subprocess.run(command, check=True)

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(Path(OUTPUT_DIR) / 'training_curves.png')))

In [ ]:
evaluation_path = Path(OUTPUT_DIR) / 'evaluation_metrics.json'
evaluation_command = [
    sys.executable, '-m', 'src_model_cls.evaluate',
    '--checkpoint', str(Path(OUTPUT_DIR) / 'best.pt'),
    '--original-data-dir', ORIGINAL_DATA_DIR,
    '--bbox-data-dir', BBOX_DATA_DIR,
    '--splits', 'val', 'test',
    '--batch-size', str(BATCH_SIZE),
    '--num-workers', str(NUM_WORKERS),
    '--json-output', str(evaluation_path),
]
subprocess.run(evaluation_command, check=True)

In [ ]:
import json
import pandas as pd

with evaluation_path.open() as file:
    results = json.load(file)
pd.DataFrame(results).T[['loss', 'exact_match', 'binary_match']]